In [ ]:
# [1] Install  (Runtime > Change runtime type > T4 GPU first)
!pip install -q spacy pyyaml networkx transformers accelerate openpyxl bitsandbytes
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 63.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# [2] spaCy + GPU check
import spacy, torch
nlp = spacy.load("en_core_web_sm")
print("spaCy", spacy.__version__, "| GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"

spaCy 3.8.16 | GPU: True


In [ ]:
# [3] Google Drive (LLM cache lives here so it survives disconnects)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# [4] Config + schema
import yaml
from dataclasses import dataclass, field
from typing import Optional

CONFIG = yaml.safe_load("""
weights: {rule: 0.45, glossary: 0.20, support: 0.20, llm: 0.15}
rule_conf: {R0: 0.95, R1: 0.70, R2: 0.70, R3: 0.80}
support_cap: 3
single_source_penalty: 0.8
theta_accept: 0.60
theta_low: 0.30
llm: {runs: 2, temperature: 0.1}
version: "0.2"
""")

@dataclass
class Span:
    story_id: str
    start: int
    end: int
    source: str

@dataclass
class Candidate:
    type: str
    name: str
    spans: list = field(default_factory=list)
    rule_ids: list = field(default_factory=list)
    source: Optional[str] = None
    target: Optional[str] = None
    cardinality_hint: Optional[str] = None
    llm_flag: bool = False
    score: float = 0.0
    status: str = "pending"
    constraint_log: list = field(default_factory=list)

In [ ]:
# [5] Rule-based path
import re

NEG = {"docker", "kafka", "api", "database", "system", "user interface"}
PRON = {"i", "we", "it", "they", "he", "she", "you"}
ROLE_RE = re.compile(r"^\s*As an? ([^,]+),", re.I)

def find_subj(verb):
    v = verb
    while True:
        subj = [c for c in v.children if c.dep_ in {"nsubj", "nsubjpass"}]
        if subj:
            return subj[0]
        if v.dep_ in {"xcomp", "advcl", "conj", "ccomp"} and v.head is not v:
            v = v.head
        else:
            return None

def rule_candidates(story_id, text):
    doc = nlp(text)
    out = []
    role, role_tok_i = None, None
    m = ROLE_RE.match(text)
    if m:
        last = [t for t in doc if t.idx + len(t.text) == m.end(1)]
        if last:
            role = last[0].lemma_.capitalize()
            role_tok_i = last[0].i
            out.append(Candidate("class", role,
                spans=[Span(story_id, m.start(1), m.end(1), "rule:R0")],
                rule_ids=["R0"]))

    def resolve(tok):
        if tok.lemma_.lower() in PRON:
            return role
        return tok.lemma_.capitalize()

    for tok in doc:
        if tok.pos_ == "NOUN" and tok.dep_ in {"nsubj", "dobj", "pobj"}:
            has_noun_mod = any(c.dep_ == "compound" for c in tok.children)
            if tok.lemma_.lower() in NEG or tok.i == role_tok_i or has_noun_mod:
                continue
            out.append(Candidate("class", tok.lemma_.capitalize(),
                spans=[Span(story_id, tok.idx, tok.idx + len(tok.text), "rule:R1")],
                rule_ids=["R1"]))
        if tok.dep_ == "compound" and tok.head.pos_ == "NOUN":
            h = tok.head
            owner = tok.lemma_.capitalize()
            span = Span(story_id, tok.idx, h.idx + len(h.text), "rule:R2")
            out.append(Candidate("attribute", h.lemma_, spans=[span],
                rule_ids=["R2"], source=owner))
            out.append(Candidate("class", owner, spans=[span], rule_ids=["R2"]))
        if tok.pos_ == "VERB":
            subj = find_subj(tok)
            obj = [c for c in tok.children if c.dep_ == "dobj"]
            if subj is not None and obj:
                src, tgt = resolve(subj), resolve(obj[0])
                if src and tgt and tok.lemma_.lower() not in {"want", "like", "need"}:
                    out.append(Candidate("relation", tok.lemma_,
                        spans=[Span(story_id, tok.idx, tok.idx + len(tok.text), "rule:R3")],
                        rule_ids=["R3"], source=src, target=tgt))
    return out

In [ ]:
# [6] Open-source LLM (Qwen2.5-3B) + prompt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"   # 4-bit (nf4) so that it fits a 16 GB T4

SYSTEM = """You extract domain-model candidates from ONE user story.
Return ONLY a JSON array. Start your answer with [ and end with ].
Each item: {"type": "class"|"attribute"|"relation",
 "evidence_quote": str (EXACT substring copied from the story),
 "name": str, "source": str|null, "target": str|null,
 "cardinality_hint": str|null, "uncertain": bool}
Rules:
- "name" MUST be a word or phrase that appears in the story (use the verb itself for relations).
- NEVER invent attributes such as id, date, status, or name unless the story says them.
- class = a business entity; attribute = a property of a class (set "source" to the owning class);
  relation = a verb linking two classes (set "source" and "target").
- If unsure, set "uncertain": true.
Example story: "As a clerk, I want to ship a parcel so that the customer gets the parcel."
Example output: [{"type":"class","evidence_quote":"parcel","name":"parcel","source":null,"target":null,"cardinality_hint":null,"uncertain":false},
{"type":"relation","evidence_quote":"ship a parcel","name":"ship","source":"clerk","target":"parcel","cardinality_hint":null,"uncertain":false}]"""

tok = AutoTokenizer.from_pretrained(MODEL)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto")

def call_llm(story_text, temperature):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"User story: {story_text}"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=700, do_sample=True,
                             temperature=max(temperature, 0.05),
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
# [7] LLM path: Drive cache, evidence + grounding checks, two-run agreement
import json, os, time, hashlib

CACHE_PATH = "/content/drive/MyDrive/exdom/qwen7b/llm_cache.jsonl"
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
_cache = {}
if os.path.exists(CACHE_PATH):
    for line in open(CACHE_PATH, encoding="utf-8"):
        d = json.loads(line)
        _cache[d["k"]] = d["raw"]
print("cache entries:", len(_cache))

def call_llm_safe(text, temperature, run_id=0):
    sh = hashlib.sha256(SYSTEM.encode()).hexdigest()[:8]
    k = hashlib.sha256(f"{MODEL}|{sh}|{temperature}|{run_id}|{text}".encode()).hexdigest()
    if k in _cache:
        return _cache[k]
    torch.manual_seed(1234 + run_id)
    raw = call_llm(text, temperature)
    _cache[k] = raw
    with open(CACHE_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": k, "model": MODEL, "prompt": sh, "T": temperature,
                            "run": run_id, "story": text, "raw": raw},
                           ensure_ascii=False) + "\n")
    return raw

VALID_TYPES = {"class", "attribute", "relation"}

def parse_json(raw):
    m = re.search(r"\[.*\]", raw.strip(), flags=re.S)
    if not m:
        return []
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    return [d for d in data if isinstance(d, dict)] if isinstance(data, list) else []

def is_grounded(name, text):
    if not name:
        return True
    story = {w for t in nlp(text) for w in (t.text.lower(), t.lemma_.lower())}
    name = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(name)).replace("_", " ")
    parts = [t.lemma_.lower() for t in nlp(name) if t.is_alpha]
    return bool(parts) and all(p in story for p in parts)

def to_candidates(story_id, text, items, stats):
    out = []
    for it in items:
        stats["proposed"] += 1
        if it.get("type") not in VALID_TYPES or not it.get("name"):
            stats["malformed"] += 1
            continue
        q = (it.get("evidence_quote") or "").strip()
        if it.get("uncertain") or not q:
            stats["uncertain"] += 1
            continue
        pos = text.lower().find(q.lower())
        if pos < 0:
            stats["invalid_evidence"] += 1
            continue
        if not all(is_grounded(x, text) for x in (it["name"], it.get("source"), it.get("target"))):
            stats["ungrounded"] += 1
            continue
        out.append(Candidate(
            it["type"], str(it["name"]).strip().capitalize(),
            spans=[Span(story_id, pos, pos + len(q), "llm")],
            source=(it.get("source") or None) and str(it["source"]).capitalize(),
            target=(it.get("target") or None) and str(it["target"]).capitalize(),
            cardinality_hint=it.get("cardinality_hint"), llm_flag=True))
    return out

def key(c):
    return (c.type, c.name.lower(), (c.source or "").lower(),
            (c.target or "").lower(), c.spans[0].start, c.spans[0].end)

STAT_KEYS = ("proposed", "malformed", "uncertain", "invalid_evidence",
             "ungrounded", "parse_fail", "kept")

def llm_candidates(story_id, text, stats=None):
    stats = stats if stats is not None else {}
    for k in STAT_KEYS:
        stats.setdefault(k, 0)
    T = CONFIG["llm"]["temperature"]
    runs = []
    for r in range(CONFIG["llm"]["runs"]):
        raw = call_llm_safe(text, T, run_id=r)
        items = parse_json(raw)
        if not items and raw.strip() != "[]":
            stats["parse_fail"] += 1
        runs.append(to_candidates(story_id, text, items, stats))
    common = set(map(key, runs[0]))
    for rr in runs[1:]:
        common &= set(map(key, rr))
    stats["kept"] += len(common)
    return [c for c in runs[0] if key(c) in common]

cache entries: 974


In [ ]:
# [8] Merge, scoring, zones
from functools import lru_cache

@lru_cache(maxsize=None)
def norm(s):
    if not s:
        return ""
    return " ".join(t.lemma_.lower() for t in nlp(s))

def merge(cands, glossary=None):
    glossary = glossary or {}
    m = lambda x: glossary.get(x, x)
    groups = {}
    for c in cands:
        n, s, t = m(norm(c.name)), m(norm(c.source)), m(norm(c.target))
        k = (c.type, n, s, t)
        if k not in groups:
            groups[k] = Candidate(c.type, n.capitalize(),
                                  source=s.capitalize() or None,
                                  target=t.capitalize() or None)
        g = groups[k]
        for sp in c.spans:
            if sp not in g.spans:
                g.spans.append(sp)
        g.rule_ids += [r for r in c.rule_ids if r not in g.rule_ids]
        g.llm_flag = g.llm_flag or c.llm_flag
        g.cardinality_hint = g.cardinality_hint or c.cardinality_hint
    return list(groups.values())

def score(c, glossary_terms=None):
    W = CONFIG["weights"]
    sig = {
        "rule": max((CONFIG["rule_conf"][r] for r in c.rule_ids), default=0.0),
        "llm": 1.0 if c.llm_flag else 0.0,
        "support": min(1.0, len({sp.story_id for sp in c.spans}) / CONFIG["support_cap"]),
    }
    if glossary_terms is not None:
        sig["glossary"] = 1.0 if c.name.lower() in glossary_terms else 0.0
    raw = sum(W[k] * sig[k] for k in sig) / sum(W[k] for k in sig)
    if bool(c.rule_ids) != c.llm_flag:
        raw *= CONFIG["single_source_penalty"]
    c.score, c.signals = round(raw, 3), sig

def overlap(a, b):
    return a.story_id == b.story_id and a.start < b.end and b.start < a.end

def set_margins(cands):
    for c in cands:
        rivals = [o for o in cands if o is not c and o.type == c.type
                  and any(overlap(a, b) for a in c.spans for b in o.spans)]
        c.margin = round(c.score - max((o.score for o in rivals), default=0.0), 3)

def zone(c):
    if c.score >= CONFIG["theta_accept"]:
        return "accept"
    return "low" if c.score < CONFIG["theta_low"] else "gray"

In [ ]:
# [9] UML constraint engine + decisions
import copy
import networkx as nx

CONFIG["include_gray"] = False
CONFIG["faith_delta"] = 0.05

CARD_RE = re.compile(r"^([01])\s*(?:\.\.|:|-|to)\s*(1|\*|n|m|many)$", re.I)
CARD_WORDS = {"one": "1", "1": "1", "many": "0..*", "*": "0..*"}
PART_VERBS = {"contain", "consist", "comprise", "include"}
LIFECYCLE_RE = re.compile(
    r"\b(is|are|gets?)\s+(also\s+)?(deleted|removed)\b|cannot exist without|only exists? (with|within)", re.I)

def parse_card(hint):
    if not hint:
        return None
    h = str(hint).strip().lower()
    if h in CARD_WORDS:
        return CARD_WORDS[h]
    m = CARD_RE.match(h)
    if m:
        hi = "1" if m.group(2) == "1" else "*"
        return f"{m.group(1)}..{hi}"
    return None

def reject(c, reason):
    c.status = "rejected"
    c.constraint_log.append(reason)

def decide(merged, texts, include_gray=None, use_constraints=True, accept_all=False):
    if include_gray is None:
        include_gray = CONFIG["include_gray"]
    decs = copy.deepcopy(merged)
    for c in decs:
        c.zone = zone(c)
        ok = accept_all or c.zone == "accept" or (c.zone == "gray" and include_gray)
        c.status = "final" if ok else "low"
    if not use_constraints:
        return decs

    by_class = {c.name.lower(): c for c in decs if c.type == "class"}
    for c in decs:
        if c.type == "relation" and c.status == "final":
            for e in (c.source, c.target):
                k = by_class.get((e or "").lower())
                if k is not None and k.status == "low" and k.zone == "gray":
                    k.status = "adjusted"
                    k.constraint_log.append(
                        f"promoted from gray zone: required endpoint of accepted relation '{c.name}'")
    classes = {n for n, k in by_class.items() if k.status in ("final", "adjusted")}

    for c in decs:
        if c.status != "final":
            continue
        if c.type == "attribute":
            if not c.source or c.source.lower() not in classes:
                reject(c, "attribute owner is not an accepted class -> rejected")
        elif c.type == "relation":
            bad = [e for e in (c.source, c.target) if not e or e.lower() not in classes]
            if bad:
                e0 = bad[0] or "?"
                k = by_class.get(e0.lower())
                why = (f"is a low-confidence class (score {k.score:.2f})" if k
                       else "is not a candidate class")
                reject(c, f"endpoint '{e0}' {why} -> rejected")
                continue
            card = parse_card(c.cardinality_hint)
            if card is None:
                c.cardinality = "0..*"
                if c.cardinality_hint:
                    c.constraint_log.append(f"hint '{c.cardinality_hint}' not valid -> default 0..*")
                else:
                    c.constraint_log.append("no cardinality evidence -> conservative default 0..*")
                c.status = "adjusted"
            else:
                c.cardinality = card
            c.kind = "association"
            if c.name.lower() in PART_VERBS:
                if any(LIFECYCLE_RE.search(texts[s]) for s in {sp.story_id for sp in c.spans}):
                    c.kind = "composition"
                else:
                    c.constraint_log.append("no lifecycle dependency evidence -> composition downgraded to association")
                    c.status = "adjusted"
    return decs

def final_items(decs):
    return [c for c in decs if c.status in ("final", "adjusted")]

def uml_checks(items):
    cs = {c.name.lower() for c in items if c.type == "class"}
    n_cls = sum(1 for c in items if c.type == "class")
    res = [("unique_class_names", n_cls == len(cs))]
    for c in items:
        if c.type == "attribute":
            res.append(("attr_owner_is_class", bool(c.source) and c.source.lower() in cs))
        elif c.type == "relation":
            for e in (c.source, c.target):
                res.append(("rel_endpoint_is_class", bool(e) and e.lower() in cs))
    return res

def find_gen_cycle(edges):
    g = nx.DiGraph(edges)
    try:
        return nx.find_cycle(g)
    except nx.NetworkXNoCycle:
        return None

assert find_gen_cycle([("A", "B"), ("B", "A")]) is not None
assert find_gen_cycle([("A", "B"), ("B", "C")]) is None
print("constraint engine ready")

constraint engine ready


In [ ]:
# [10] Explanations (style B1) with computed counterfactuals, and the trace book
import uuid, datetime

STATUS_WORD = {"final": "accepted", "adjusted": "accepted with adjustment",
               "low": "set aside as low-confidence", "rejected": "rejected"}

def drop_spans(c, drop):
    """Counterfactual: score and zone of candidate c if the given evidence spans were absent
    (computed with the real scoring function, not estimated)."""
    keep = [s for s in c.spans if s not in drop]
    if not keep:
        return 0.0, "low"
    d = copy.copy(c)
    d.spans = keep
    d.rule_ids = sorted({s.source.split(":", 1)[1] for s in keep if s.source.startswith("rule:")})
    d.llm_flag = any(s.source == "llm" for s in keep)
    score(d)
    return d.score, zone(d)

def path_ablation(c):
    """What the decision would be without each evidence path (rule path / LLM path)."""
    out = {}
    for label, group in (("rule", [s for s in c.spans if s.source.startswith("rule:")]),
                         ("LLM", [s for s in c.spans if s.source == "llm"])):
        if group:
            out[label] = drop_spans(c, group)
    return out

def decisive(c, s1, z1):
    return (c.score - s1) >= CONFIG["faith_delta"] or z1 != zone(c)

def pick_spans(c, k=3):
    """Spans to quote: first one per evidence source (rule id / LLM), then spans from new stories."""
    picked = []
    for s in c.spans:
        if s.source not in {p.source for p in picked}:
            picked.append(s)
    for s in c.spans:
        if len(picked) >= k:
            break
        if s not in picked and s.story_id not in {p.story_id for p in picked}:
            picked.append(s)
    return picked[:k]

def explain_b1(c, texts):
    label = c.name
    if c.type == "relation":
        label = f"{c.source} --{c.name}--> {c.target}"
    elif c.type == "attribute":
        label = f"{c.source}.{c.name}"
    cited = pick_spans(c)
    parts = [f"{c.type.capitalize()} '{label}' was {STATUS_WORD[c.status]}."]
    q = "; ".join(f'[{s.story_id}] "{texts[s.story_id][s.start:s.end]}"' for s in cited)
    parts.append(f"Evidence: {q}.")
    sig = []
    if c.rule_ids:
        sig.append("rules " + ", ".join(c.rule_ids))
    if c.llm_flag:
        sig.append("LLM suggestion with verified quote")
    n_st = len({s.story_id for s in c.spans})
    parts.append(f"Active signals: {'; '.join(sig) or 'none'}; score {c.score:.2f} ({c.zone}); "
                 f"{len(c.spans)} evidence spans in {n_st} stories.")
    ab = path_ablation(c)
    if ab:
        cf = "; ".join(f"without the {k} evidence: {s1:.2f} ({z1})" for k, (s1, z1) in ab.items())
        parts.append(f"Counterfactual: {cf}.")
    parts.append("Structural checks: " + ("; ".join(c.constraint_log) if c.constraint_log else "passed") + ".")
    if c.status == "low":
        gap = max(0.0, CONFIG["theta_accept"] - c.score)
        parts.append(f"To be accepted it needs about +{gap:.2f}: confirmation by the second path "
                     f"(rule match or LLM-verified quote) or support from more stories.")
    if c.status == "rejected":
        parts.append("To admit it, the endpoint class named above would first need enough evidence to be accepted.")
    text = " ".join(parts)
    named = set(re.findall(r"\bR\d\b", text))
    quotes = re.findall(r'\[(\S+?)\] "([^"]*)"', text)
    ok = named <= set(c.rule_ids) and all(qq in texts.get(sid, "") for sid, qq in quotes)
    return {"text": text, "fallback": False, "verified": ok}

TB_PATH = "/content/tracebook.jsonl"

def write_tracebook(decs, expls, path=TB_PATH):
    now = datetime.datetime.now(datetime.timezone.utc).isoformat()
    with open(path, "w", encoding="utf-8") as f:
        for c, e in zip(decs, expls):
            rec = {"id": str(uuid.uuid4()), "type": c.type, "name": c.name,
                   "source": c.source, "target": c.target, "status": c.status,
                   "score": c.score, "margin": getattr(c, "margin", None),
                   "zone": getattr(c, "zone", None), "signals": getattr(c, "signals", {}),
                   "rule_ids": c.rule_ids, "llm": c.llm_flag,
                   "constraint_log": c.constraint_log,
                   "cardinality": getattr(c, "cardinality", None), "kind": getattr(c, "kind", None),
                   "evidence": [[s.story_id, s.start, s.end, s.source] for s in c.spans],
                   "config_version": CONFIG["version"], "timestamp": now,
                   "explanation_verified": e["verified"]}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def elements_for_span(story_id, start, end, path=TB_PATH):
    out = []
    for line in open(path, encoding="utf-8"):
        r = json.loads(line)
        if any(e[0] == story_id and e[1] < end and start < e[2] for e in r["evidence"]):
            out.append((r["type"], r["name"], r["status"]))
    return out

In [ ]:
# [11] Download benchmark (Bragilovski et al., Zenodo 15223349)
!wget -q -O REJ-main.zip "https://zenodo.org/records/15223349/files/REJ-main.zip?download=1"
!md5sum REJ-main.zip
!unzip -q -o REJ-main.zip -d REJ

f628b349b6f9cb9f6f196fce5d35a899  REJ-main.zip


In [ ]:
import glob, os, re
import pandas as pd
ROOT = "REJ/REJ-main/A Benchmark for Domain Model"
key = lambda n: re.sub(r"[^a-z0-9]", "", str(n).lower())
st = {}
for f in glob.glob(f"{ROOT}/User Stories/*.txt"):
    st[key(os.path.splitext(os.path.basename(f))[0])] = sum(1 for l in open(f, encoding="utf-8-sig", errors="replace") if l.strip())
gc = pd.read_excel(f"{ROOT}/Domain Models/Gold standard - Classes.xlsx", header=0)
cls = {key(c): int(gc[c].dropna().astype(str).str.strip().ne("").sum()) for c in gc.columns}
ga = pd.read_excel(f"{ROOT}/Domain Models/Gold standard - Associations.xlsx", header=None)
asc = {}
for j in range(0, ga.shape[1], 2):
    if pd.isna(ga.iloc[0, j]): continue
    asc[key(ga.iloc[0, j])] = sum(pd.notna(ga.iloc[i, j]) and pd.notna(ga.iloc[i, j+1]) for i in range(2, ga.shape[0]))
for p in sorted(st): print(p, st[p], cls.get(p), asc.get(p))

In [ ]:
# [12] Benchmark loader, name matching, metrics, pools, conditions
import glob, random, itertools, os
import pandas as pd

ROOT = "REJ/REJ-main"
BASE = f"{ROOT}/A Benchmark for Domain Model"
PROJECTS = {}

def key_of(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

def load_benchmark(base=BASE):
    st = {}
    for f in glob.glob(f"{base}/User Stories/*.txt"):
        k = key_of(os.path.splitext(os.path.basename(f))[0])
        with open(f, encoding="utf-8-sig", errors="replace") as fh:
            st[k] = [l.strip() for l in fh.read().splitlines() if l.strip()]
    gc = pd.read_excel(f"{base}/Domain Models/Gold standard - Classes.xlsx", header=0)
    classes = {key_of(c): [str(v).strip() for v in gc[c].dropna() if str(v).strip()]
               for c in gc.columns}
    ga = pd.read_excel(f"{base}/Domain Models/Gold standard - Associations.xlsx", header=None)
    assoc = {}
    for j in range(0, ga.shape[1], 2):
        if pd.isna(ga.iloc[0, j]):
            continue
        pairs = []
        for i in range(2, ga.shape[0]):
            x, y = ga.iloc[i, j], ga.iloc[i, j + 1]
            if pd.notna(x) and pd.notna(y):
                pairs.append([str(x).strip(), str(y).strip()])
        assoc[key_of(ga.iloc[0, j])] = pairs
    for pid, lines in st.items():
        PROJECTS[pid] = {
            "stories": {f"{pid}-{i+1}": t for i, t in enumerate(lines)},
            "gold": {"classes": classes[pid], "attributes": [], "relations": assoc[pid]}}
    return sorted(st)

BM = load_benchmark()
for pid in BM:
    g = PROJECTS[pid]["gold"]
    print(f"{pid:14} stories={len(PROJECTS[pid]['stories']):3}  gold classes={len(g['classes']):2}  gold assoc={len(g['relations']):2}")

@lru_cache(maxsize=None)
def nrm(s):
    return norm(s) if s else ""

def match_names(a, b, relaxed):
    a, b = nrm(a), nrm(b)
    if not a or not b:
        return False
    if a == b:
        return True
    if not relaxed:
        return False
    ta, tb = a.split(), b.split()
    return ta[-1] == tb[-1] and (set(ta) <= set(tb) or set(tb) <= set(ta))

def same_pair(p, q, relaxed):
    return ((match_names(p[0], q[0], relaxed) and match_names(p[1], q[1], relaxed)) or
            (match_names(p[0], q[1], relaxed) and match_names(p[1], q[0], relaxed)))

def prf(model, gold, same):
    M, G = list(model), list(gold)
    P = sum(any(same(m, g) for g in G) for m in M) / len(M) if M else 0.0
    R = sum(any(same(m, g) for m in M) for g in G) / len(G) if G else 0.0
    out = {"P": P, "R": R}
    for b in (0.5, 1, 2):
        out[f"F{b}"] = (1 + b * b) * P * R / (b * b * P + R) if P + R else 0.0
    return out

def split_ids(ids, test_frac=0.5, seed=13):
    ids = sorted(ids)
    random.Random(seed).shuffle(ids)
    k = max(1, int(len(ids) * test_frac))
    return ids[k:], ids[:k]          # dev, test

POOL = {}
def build_pool(pid):
    rule, llm, st = [], [], {}
    for sid, t in PROJECTS[pid]["stories"].items():
        rule += rule_candidates(sid, t)
        llm += llm_candidates(sid, t, st)
    POOL[pid] = (rule, llm, st)

def build_pool_rules(pid):
    rule = []
    for sid, t in PROJECTS[pid]["stories"].items():
        rule += rule_candidates(sid, t)
    old = POOL.get(pid, (None, [], {}))
    POOL[pid] = (rule, old[1], old[2])

def run_condition(pid, use_rule, use_llm, include_gray, use_constraints, accept_all=False):
    rule, llm, _ = POOL[pid]
    merged_ = merge((rule if use_rule else []) + (llm if use_llm else []))
    for c in merged_:
        score(c)
    set_margins(merged_)
    decs_ = decide(merged_, PROJECTS[pid]["stories"], include_gray, use_constraints, accept_all)
    return final_items(decs_), decs_

CONDITIONS = {
    "A1 rules (raw)":           dict(use_rule=True,  use_llm=False, include_gray=True,  use_constraints=False, accept_all=True),
    "A3 LLM (raw)":             dict(use_rule=False, use_llm=True,  include_gray=True,  use_constraints=False, accept_all=True),
    "A1 + constraints":         dict(use_rule=True,  use_llm=False, include_gray=True,  use_constraints=True,  accept_all=True),
    "A3 + constraints":         dict(use_rule=False, use_llm=True,  include_gray=True,  use_constraints=True,  accept_all=True),
    "EXDOM precision mode":     dict(use_rule=True,  use_llm=True,  include_gray=False, use_constraints=True),
    "EXDOM recall mode":        dict(use_rule=True,  use_llm=True,  include_gray=True,  use_constraints=True),
    "EXDOM recall, no constr.": dict(use_rule=True,  use_llm=True,  include_gray=True,  use_constraints=False),
}

def evaluate_bm(pids, relaxed=True, conds=None):
    rows = []
    for name, cfg in (conds or CONDITIONS).items():
        for pid in pids:
            model_, _ = run_condition(pid, **cfg)
            Mc = sorted({c.name for c in model_ if c.type == "class"})
            Mp = sorted({tuple(sorted((c.source, c.target))) for c in model_
                         if c.type == "relation" and c.source and c.target})
            g = PROJECTS[pid]["gold"]
            rc = prf(Mc, g["classes"], lambda a, b: match_names(a, b, relaxed))
            ra = prf(Mp, g["relations"], lambda p, q: same_pair(p, q, relaxed))
            rows.append({"condition": name, "project": pid,
                         **{f"cls_{k}": v for k, v in rc.items()},
                         **{f"asc_{k}": v for k, v in ra.items()}})
    df = pd.DataFrame(rows)
    return df.groupby("condition", sort=False).mean(numeric_only=True).round(3)

def eval_macro(pids, cfg, relaxed=True):
    cs, asc = [], []
    for pid in pids:
        model_, _ = run_condition(pid, **cfg)
        Mc = sorted({c.name for c in model_ if c.type == "class"})
        Mp = sorted({tuple(sorted((c.source, c.target))) for c in model_
                     if c.type == "relation" and c.source and c.target})
        g = PROJECTS[pid]["gold"]
        cs.append(prf(Mc, g["classes"], lambda a, b: match_names(a, b, relaxed)))
        asc.append(prf(Mp, g["relations"], lambda p, q: same_pair(p, q, relaxed)))
    m = lambda L, k: sum(x[k] for x in L) / len(L)
    return {"cls_P": m(cs, "P"), "cls_R": m(cs, "R"), "cls_F1": m(cs, "F1"),
            "asc_P": m(asc, "P"), "asc_R": m(asc, "R"), "asc_F1": m(asc, "F1")}

def grid_full(pids):
    rows = []
    keys = ("weights", "single_source_penalty", "theta_accept", "theta_low", "support_cap")
    old = {k: copy.deepcopy(CONFIG[k]) for k in keys}
    for cap, pen, wl, ta, tl in itertools.product(
            (5, 12), (0.6, 0.8, 1.0), (0.15, 0.30), (0.5, 0.6), (0.3, 0.4)):
        CONFIG.update(support_cap=cap, single_source_penalty=pen, theta_accept=ta, theta_low=tl)
        CONFIG["weights"] = {**old["weights"], "llm": wl}
        for mode, gray in (("precision", False), ("recall", True)):
            r = eval_macro(pids, dict(use_rule=True, use_llm=True,
                                      include_gray=gray, use_constraints=True))
            rows.append({"cap": cap, "pen": pen, "w_llm": wl, "t_acc": ta, "t_low": tl,
                         "mode": mode, **{k: round(v, 3) for k, v in r.items()}})
    CONFIG.update(old)
    return pd.DataFrame(rows)

def llm_report(pids):
    tot = {k: sum(POOL[p][2].get(k, 0) for p in pids) for k in STAT_KEYS}
    prop = tot["proposed"] or 1
    print(tot)
    print("invalid-evidence rate:", round(tot["invalid_evidence"] / prop, 3),
          "| ungrounded-name rate:", round(tot["ungrounded"] / prop, 3))

def candidate_precision_by_source(pids, relaxed=True):
    rows = []
    for pid in pids:
        rule, llm, _ = POOL[pid]
        gold = PROJECTS[pid]["gold"]["classes"]
        for c in merge(rule + llm):
            if c.type != "class":
                continue
            src = "both" if (c.rule_ids and c.llm_flag) else ("rule only" if c.rule_ids else "llm only")
            n = len({sp.story_id for sp in c.spans})
            hit = any(match_names(c.name, g, relaxed) for g in gold)
            rows.append({"pid": pid, "src": src, "n_stories": n, "hit": hit})
    df = pd.DataFrame(rows)
    print(df.groupby("src")["hit"].agg(n="count", precision="mean").round(3))
    df["support"] = pd.cut(df["n_stories"], [0, 1, 2, 4, 9, 1000],
                           labels=["1", "2", "3-4", "5-9", "10+"])
    print(df.groupby(["src", "support"], observed=True)["hit"]
            .agg(n="count", precision="mean").round(3))

camperplus     stories= 55  gold classes=17  gold assoc=23
fishchips      stories= 50  gold classes= 5  gold assoc= 8
grocery        stories= 49  gold classes= 9  gold assoc= 8
planningpoker  stories= 53  gold classes= 6  gold assoc= 6
recycling      stories= 51  gold classes= 9  gold assoc= 6
school         stories= 61  gold classes=17  gold assoc=23
sports         stories= 63  gold classes=13  gold assoc=12
supermarket    stories= 51  gold classes=11  gold assoc=13
ticket         stories= 54  gold classes= 9  gold assoc=13


In [ ]:
# [13] Fixed dev/test split (by project, seed 13) + rule candidate coverage on DEV
DEV, TEST = split_ids(BM, test_frac=0.5, seed=13)
print("DEV :", DEV)
print("TEST:", TEST)
assert DEV == ['fishchips', 'sports', 'school', 'ticket', 'recycling'], "split changed!"

def rule_coverage(pids):
    for pid in pids:
        P = PROJECTS[pid]
        cand = set()
        for sid, t in P["stories"].items():
            cand |= {c.name for c in rule_candidates(sid, t) if c.type == "class"}
        gold = P["gold"]["classes"]
        hs = sum(any(match_names(c, g, False) for c in cand) for g in gold)
        hr = sum(any(match_names(c, g, True) for c in cand) for g in gold)
        print(f"{pid:14} gold={len(gold):2}  rule class cands={len(cand):3}  "
              f"reached strict={hs}/{len(gold)}  relaxed={hr}/{len(gold)}")

print("\n-- rule candidate coverage (DEV only) --")
rule_coverage(DEV)

DEV : ['fishchips', 'sports', 'school', 'ticket', 'recycling']
TEST: ['planningpoker', 'camperplus', 'grocery', 'supermarket']

-- rule candidate coverage (DEV only) --
fishchips      gold= 5  rule class cands= 61  reached strict=4/5  relaxed=4/5
sports         gold=13  rule class cands= 52  reached strict=12/13  relaxed=13/13
school         gold=17  rule class cands= 79  reached strict=15/17  relaxed=16/17
ticket         gold= 9  rule class cands= 69  reached strict=8/9  relaxed=9/9
recycling      gold= 9  rule class cands=100  reached strict=5/9  relaxed=8/9


In [ ]:
# [14] Build candidate pools on DEV (rules + LLM; LLM answers are cached on Drive)
t0 = time.time()
for pid in DEV:
    build_pool(pid)
    print(f"{pid:12} done | {POOL[pid][2]} | {(time.time()-t0)/60:.1f} min")
llm_report(DEV)

fishchips    done | {'proposed': 384, 'malformed': 0, 'uncertain': 27, 'invalid_evidence': 6, 'ungrounded': 18, 'parse_fail': 4, 'kept': 155} | 0.1 min
sports       done | {'proposed': 388, 'malformed': 0, 'uncertain': 25, 'invalid_evidence': 2, 'ungrounded': 24, 'parse_fail': 6, 'kept': 156} | 0.2 min
school       done | {'proposed': 479, 'malformed': 0, 'uncertain': 27, 'invalid_evidence': 12, 'ungrounded': 32, 'parse_fail': 16, 'kept': 183} | 0.4 min
ticket       done | {'proposed': 336, 'malformed': 0, 'uncertain': 12, 'invalid_evidence': 11, 'ungrounded': 29, 'parse_fail': 18, 'kept': 128} | 0.5 min
recycling    done | {'proposed': 280, 'malformed': 0, 'uncertain': 19, 'invalid_evidence': 4, 'ungrounded': 45, 'parse_fail': 23, 'kept': 84} | 0.6 min
{'proposed': 1867, 'malformed': 0, 'uncertain': 110, 'invalid_evidence': 35, 'ungrounded': 148, 'parse_fail': 67, 'kept': 706}
invalid-evidence rate: 0.019 | ungrounded-name rate: 0.079


In [ ]:
# [15] Default-setting evaluation on DEV + where the LLM adds signal
print("-- relaxed --")
print(evaluate_bm(DEV, True).to_string())
print("\n-- strict --")
print(evaluate_bm(DEV, False).to_string())
print("\n-- candidate precision by source and support --")
candidate_precision_by_source(DEV)

-- relaxed --
                          cls_P  cls_R  cls_F0.5  cls_F1  cls_F2  asc_P  asc_R  asc_F0.5  asc_F1  asc_F2
condition                                                                                               
A1 rules (raw)            0.147  0.926     0.175   0.248   0.430  0.057  0.267     0.067   0.092   0.146
A3 LLM (raw)              0.253  0.589     0.283   0.345   0.452  0.087  0.217     0.097   0.118   0.158
A1 + constraints          0.147  0.926     0.175   0.248   0.430  0.066  0.267     0.077   0.103   0.159
A3 + constraints          0.253  0.589     0.283   0.345   0.452  0.098  0.217     0.108   0.129   0.165
EXDOM precision mode      0.368  0.589     0.395   0.447   0.520  0.204  0.124     0.179   0.153   0.134
EXDOM recall mode         0.144  0.926     0.173   0.244   0.425  0.066  0.267     0.077   0.103   0.159
EXDOM recall, no constr.  0.144  0.926     0.173   0.244   0.425  0.057  0.267     0.067   0.092   0.146

-- strict --
                          c

In [ ]:
# [16] OPTIONAL: sensitivity grid on DEV (takes a few minutes)
# G = grid_full(DEV)
# print(G.sort_values("cls_F1", ascending=False).head(10).to_string(index=False))
# print(G.groupby("mode")["cls_F1"].describe().round(3))

In [ ]:
# [17] Final-stage helpers: output folder, tuning grid, cached run_condition
import itertools, collections, hashlib, json, datetime, copy

OUT = "/content/drive/MyDrive/exdom/qwen7b"
os.makedirs(OUT, exist_ok=True)

BASE_W = {"rule": 0.45, "glossary": 0.20, "support": 0.20, "llm": 0.15}
GRID = dict(w_rule=(0.15, 0.30, 0.45), w_llm=(0.15, 0.30, 0.45), pen=(0.8, 1.0),
            cap=(5, 12), t_acc=(0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60))

_MERGED = {}     # merge() does not depend on weights, so cache it per (project, paths)
def run_condition(pid, use_rule, use_llm, include_gray, use_constraints, accept_all=False):
    k = (pid, use_rule, use_llm)
    if k not in _MERGED:
        rule, llm, _ = POOL[pid]
        _MERGED[k] = merge((rule if use_rule else []) + (llm if use_llm else []))
    merged_ = _MERGED[k]
    for c in merged_:
        score(c)
    decs_ = decide(merged_, PROJECTS[pid]["stories"], include_gray, use_constraints, accept_all)
    return final_items(decs_), decs_

def apply_cfg(c):
    CONFIG.update(support_cap=c["cap"], single_source_penalty=c["pen"],
                  theta_accept=c["t_acc"], theta_low=0.30)
    CONFIG["weights"] = {**BASE_W, "rule": c["w_rule"], "llm": c["w_llm"]}

def tune(flags, pids):
    """argmax of macro class-F1 over GRID on the given projects (DEV only)."""
    snap = copy.deepcopy(CONFIG)
    cfg = dict(include_gray=False, use_constraints=True, **flags)
    rows = []
    for vals in itertools.product(*GRID.values()):
        c = dict(zip(GRID, vals))
        apply_cfg(c)
        for pid in pids:
            rows.append({**c, "pid": pid, "F1": eval_macro([pid], cfg)["cls_F1"]})
    CONFIG.clear(); CONFIG.update(snap)
    piv = pd.DataFrame(rows).pivot_table(index=list(GRID), columns="pid", values="F1")
    macro = piv.mean(axis=1)
    best = {k: float(v) for k, v in zip(GRID, macro.idxmax())}
    return best, float(macro.max()), piv

def lopo_mean(piv, pids):
    vals = []
    for held in pids:
        train = [p for p in pids if p != held]
        vals.append(float(piv.loc[piv[train].mean(axis=1).idxmax(), held]))
    return sum(vals) / len(vals)

In [ ]:
# [18] FREEZE the configuration (tuned on DEV only; TEST is untouched)
CHECK_AGAINST_PREVIOUS = False   # the expected values belong to the 3B model   # guards against pipeline drift since the earlier DEV analysis
EXPECTED = {"rules + scoring": (0.427, 0.395),   # (best macro F1, LOPO mean) from the DEV analysis
            "LLM + scoring":   (0.431, 0.396),
            "EXDOM (both)":    (0.517, 0.470)}

for pid in DEV:
    if pid not in POOL:
        build_pool(pid)          # LLM answers come from the Drive cache
_MERGED.clear()

variants = {"rules + scoring": dict(use_rule=True,  use_llm=False),
            "LLM + scoring":   dict(use_rule=False, use_llm=True),
            "EXDOM (both)":    dict(use_rule=True,  use_llm=True)}
FROZEN = {}
for name, flags in variants.items():
    best, mbest, piv = tune(flags, DEV)
    FROZEN[name] = {"config": best, "dev_macro_F1": round(mbest, 3),
                    "dev_lopo_F1": round(lopo_mean(piv, DEV), 3)}
    print(f"{name:16}", FROZEN[name])

if CHECK_AGAINST_PREVIOUS:
    for name, (opt, lo) in EXPECTED.items():
        got = FROZEN[name]
        assert abs(got["dev_macro_F1"] - opt) <= 0.003 and abs(got["dev_lopo_F1"] - lo) <= 0.003, \
            f"pipeline drift for '{name}': expected {opt}/{lo}, got {got}. Do NOT run TEST; investigate."
    print("\nDEV numbers reproduce the earlier analysis.")

CONFIG["version"] = "1.0"
frozen = {"frozen_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "pipeline_version": CONFIG["version"],
          "llm": {"model": MODEL, "temperature": CONFIG["llm"]["temperature"],
                  "runs": CONFIG["llm"]["runs"], "seed_base": 1234,
                  "prompt_sha8": hashlib.sha256(SYSTEM.encode()).hexdigest()[:8]},
          "rule_conf": CONFIG["rule_conf"], "base_weights": BASE_W, "grid": GRID,
          "split": {"seed": 13, "dev": DEV, "test": TEST},
          "selection": "argmax of macro class-F1 (relaxed matching) over the grid, on DEV only",
          "variants": FROZEN}
json.dump(frozen, open(f"{OUT}/frozen_config.json", "w", encoding="utf-8"), indent=1)
FROZEN_SHA = hashlib.sha256(json.dumps(FROZEN, sort_keys=True).encode()).hexdigest()[:12]
print("frozen config hash:", FROZEN_SHA, "| saved to", f"{OUT}/frozen_config.json")

rules + scoring  {'config': {'w_rule': 0.15, 'w_llm': 0.15, 'pen': 1.0, 'cap': 5.0, 't_acc': 0.45}, 'dev_macro_F1': 0.427, 'dev_lopo_F1': 0.395}
LLM + scoring    {'config': {'w_rule': 0.3, 'w_llm': 0.15, 'pen': 1.0, 'cap': 5.0, 't_acc': 0.3}, 'dev_macro_F1': 0.441, 'dev_lopo_F1': 0.436}
EXDOM (both)     {'config': {'w_rule': 0.15, 'w_llm': 0.15, 'pen': 0.8, 'cap': 12.0, 't_acc': 0.6}, 'dev_macro_F1': 0.512, 'dev_lopo_F1': 0.512}
frozen config hash: 4c8fbc85ad70 | saved to /content/drive/MyDrive/exdom/qwen7b/frozen_config.json


In [ ]:
# [19] ONE-SHOT evaluation on TEST. Check the frozen config above, then set CONFIRM_RUN_TEST = True.
CONFIRM_RUN_TEST = True

RESULT_PATH = f"{OUT}/TEST_RESULTS.json"
assert "FROZEN_SHA" in globals(), "run cell 18 first"

TEST_CONDS = {
    "A1 rules (raw)":        (None,              dict(use_rule=True,  use_llm=False, include_gray=True,  use_constraints=False, accept_all=True)),
    "A3 LLM (raw)":          (None,              dict(use_rule=False, use_llm=True,  include_gray=True,  use_constraints=False, accept_all=True)),
    "rules + scoring":       ("rules + scoring", dict(use_rule=True,  use_llm=False, include_gray=False, use_constraints=True)),
    "LLM + scoring":         ("LLM + scoring",   dict(use_rule=False, use_llm=True,  include_gray=False, use_constraints=True)),
    "EXDOM (both)":          ("EXDOM (both)",    dict(use_rule=True,  use_llm=True,  include_gray=False, use_constraints=True)),
    "EXDOM, no constraints": ("EXDOM (both)",    dict(use_rule=True,  use_llm=True,  include_gray=False, use_constraints=False)),
}

def eval_conditions(pids, relaxed):
    snap = copy.deepcopy(CONFIG)
    rows = []
    for name, (var, cfg) in TEST_CONDS.items():
        CONFIG.clear(); CONFIG.update(snap)
        if var:
            apply_cfg(FROZEN[var]["config"])
        for pid in pids:
            model_, _ = run_condition(pid, **cfg)
            chk = uml_checks(model_)
            Mc = sorted({c.name for c in model_ if c.type == "class"})
            Mp = sorted({tuple(sorted((c.source, c.target))) for c in model_
                         if c.type == "relation" and c.source and c.target})
            g = PROJECTS[pid]["gold"]
            rc = prf(Mc, g["classes"], lambda a, b: match_names(a, b, relaxed))
            ra = prf(Mp, g["relations"], lambda p, q: same_pair(p, q, relaxed))
            rows.append({"condition": name, "pid": pid, "n_classes": len(Mc), "n_pairs": len(Mp),
                         "uml_fail": sum(not ok for _, ok in chk), "uml_total": len(chk),
                         **{f"cls_{k}": v for k, v in rc.items()},
                         **{f"asc_{k}": v for k, v in ra.items()}})
    CONFIG.clear(); CONFIG.update(snap)
    return pd.DataFrame(rows)

def macro_table(df, cols):
    g = df.groupby("condition", sort=False)
    t = g[cols].mean()
    t["uml_fail"] = g["uml_fail"].sum()
    t["uml_checks"] = g["uml_total"].sum()
    return t.round(3)

def show_results(rel, strict):
    cols = ["cls_P", "cls_R", "cls_F1", "asc_P", "asc_R", "asc_F1", "n_classes", "n_pairs"]
    for label, df in (("relaxed", rel), ("strict", strict)):
        print(f"\n== TEST, {label} matching (macro over {df['pid'].nunique()} projects) ==")
        print(macro_table(df, cols).to_string())
    pv = rel.pivot(index="condition", columns="pid", values="cls_F1").reindex(list(TEST_CONDS))
    print("\n== per-project class F1 (relaxed) ==")
    print(pv.round(3).to_string())
    print()
    for other in ("A1 rules (raw)", "A3 LLM (raw)", "rules + scoring", "LLM + scoring"):
        d = pv.loc["EXDOM (both)"] - pv.loc[other]
        print(f"EXDOM - {other:16}: " + "  ".join(f"{p}:{v:+.3f}" for p, v in d.items())
              + f"  | wins {(d > 0).sum()}/{len(d)}")

rel = strict = None
if os.path.exists(RESULT_PATH):
    saved = json.load(open(RESULT_PATH, encoding="utf-8"))
    assert saved["frozen_hash"] == FROZEN_SHA, "saved TEST results come from a different frozen config"
    rel, strict = pd.DataFrame(saved["results_relaxed"]), pd.DataFrame(saved["results_strict"])
    print("TEST was already run once -> showing the saved results (nothing re-run).")
elif not CONFIRM_RUN_TEST:
    print("TEST not run yet. Check the frozen config from cell 18, then set CONFIRM_RUN_TEST = True "
          "and run this cell ONCE.")
else:
    t0 = time.time()
    for pid in TEST:
        if pid not in POOL:
            build_pool(pid)
        print(f"{pid:14} pool built | {POOL[pid][2]} | {(time.time()-t0)/60:.1f} min", flush=True)
    _MERGED.clear()
    llm_report(TEST)
    rel = eval_conditions(TEST, True)
    strict = eval_conditions(TEST, False)
    payload = {"run_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
               "frozen_hash": FROZEN_SHA, "pipeline_version": CONFIG["version"],
               "test_projects": TEST,
               "n_stories": {p: len(PROJECTS[p]["stories"]) for p in TEST},
               "llm_stats": {p: POOL[p][2] for p in TEST},
               "results_relaxed": rel.to_dict("records"),
               "results_strict": strict.to_dict("records")}
    json.dump(payload, open(RESULT_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))
    print("saved ->", RESULT_PATH)

if rel is not None:
    show_results(rel, strict)

planningpoker  pool built | {'proposed': 253, 'malformed': 0, 'uncertain': 24, 'invalid_evidence': 4, 'ungrounded': 7, 'parse_fail': 35, 'kept': 97} | 0.2 min
camperplus     pool built | {'proposed': 351, 'malformed': 0, 'uncertain': 15, 'invalid_evidence': 10, 'ungrounded': 17, 'parse_fail': 10, 'kept': 136} | 17.7 min
grocery        pool built | {'proposed': 231, 'malformed': 0, 'uncertain': 11, 'invalid_evidence': 5, 'ungrounded': 11, 'parse_fail': 23, 'kept': 90} | 30.3 min
supermarket    pool built | {'proposed': 356, 'malformed': 0, 'uncertain': 21, 'invalid_evidence': 5, 'ungrounded': 35, 'parse_fail': 2, 'kept': 139} | 45.6 min
{'proposed': 1191, 'malformed': 0, 'uncertain': 71, 'invalid_evidence': 24, 'ungrounded': 70, 'parse_fail': 70, 'kept': 462}
invalid-evidence rate: 0.02 | ungrounded-name rate: 0.059
saved -> /content/drive/MyDrive/exdom/qwen7b/TEST_RESULTS.json

== TEST, relaxed matching (macro over 4 projects) ==
                       cls_P  cls_R  cls_F1  asc_P  asc_

In [ ]:
# [20] Explanation and trace-book statistics on TEST (process measures, not a human evaluation)
for pid in TEST:
    if pid not in POOL:
        build_pool(pid)          # from the Drive cache

def explanation_stats(pids):
    snap = copy.deepcopy(CONFIG)
    apply_cfg(FROZEN["EXDOM (both)"]["config"])
    rows, logs, samples = [], collections.Counter(), {}
    for pid in pids:
        texts = PROJECTS[pid]["stories"]
        rule, llm, _ = POOL[pid]
        merged_ = merge(rule + llm)
        for c in merged_:
            score(c)
        set_margins(merged_)
        decs_ = decide(merged_, texts, include_gray=False, use_constraints=True)
        expls = [explain_b1(c, texts) for c in decs_]
        write_tracebook(decs_, expls, path=f"{OUT}/tracebook_{pid}.jsonl")
        for c, e in zip(decs_, expls):
            ab = path_ablation(c)
            rows.append({"pid": pid, "type": c.type, "status": c.status,
                         "n_spans": len(c.spans), "n_stories": len({s.story_id for s in c.spans}),
                         "has_rule": "rule" in ab, "has_llm": "LLM" in ab,
                         "dec_rule": bool(decisive(c, *ab["rule"])) if "rule" in ab else False,
                         "dec_llm": bool(decisive(c, *ab["LLM"])) if "LLM" in ab else False,
                         "verified": e["verified"]})
            for line in c.constraint_log:
                line = re.sub(r"'[^']*'", "'..'", line)
                logs[re.sub(r"\(score [0-9.]+\)", "(score ..)", line)] += 1
            if c.status not in samples:
                samples[c.status] = e["text"]
    CONFIG.clear(); CONFIG.update(snap)
    return pd.DataFrame(rows), logs, samples

df_e, logs, samples = explanation_stats(TEST)
acc = df_e[df_e.status.isin(["final", "adjusted"])]
both = acc[acc.has_rule & acc.has_llm]
only_rule = acc[acc.has_rule & ~acc.has_llm]
only_llm = acc[~acc.has_rule & acc.has_llm]

print("decisions explained:", len(df_e))
print(df_e.groupby(["type", "status"]).size().unstack(fill_value=0).to_string())
print("\nconsistency check passed (true by construction, sanity only):", round(df_e.verified.mean(), 3))
print("evidence spans per decision: mean", round(df_e.n_spans.mean(), 2), "| median", df_e.n_spans.median())
print("\naccepted elements:", len(acc))
print(f"  evidence from both paths: {len(both)} | rules only: {len(only_rule)} | LLM only: {len(only_llm)}")
if len(both):
    print(f"  among 'both': removing the RULE evidence changes the outcome in {both.dec_rule.mean():.0%}")
    print(f"  among 'both': removing the LLM evidence changes the outcome in {both.dec_llm.mean():.0%}")
print(f"  supported by >= 2 stories: {(acc.n_stories >= 2).mean():.0%} | by >= 5 stories: {(acc.n_stories >= 5).mean():.0%}")
print("\nconstraint-engine actions on TEST:")
for k, v in logs.most_common():
    print(f"  {v:5d}  {k}")

rng = random.Random(0)
print("\ntrace book reverse lookup (sampled; consistent by construction):")
for pid in TEST:
    path = f"{OUT}/tracebook_{pid}.jsonl"
    recs = [json.loads(l) for l in open(path, encoding="utf-8")]
    spans = [(r["type"], r["name"], r["status"], e) for r in recs for e in r["evidence"]]
    sample = rng.sample(spans, min(200, len(spans)))
    ok = sum((t, n, s) in elements_for_span(e[0], e[1], e[2], path) for t, n, s, e in sample)
    print(f"  {pid:14} {len(recs)} records | {ok}/{len(sample)} sampled spans resolve to their element")

print("\n== example explanations (one per status) ==")
for st, txt in samples.items():
    print(f"[{st}] {txt}\n")

summary = {"n_decisions": int(len(df_e)),
           "by_type_status": {f"{t}/{s}": int(n) for (t, s), n in df_e.groupby(["type", "status"]).size().items()},
           "accepted": int(len(acc)), "accepted_both_paths": int(len(both)),
           "accepted_rule_only": int(len(only_rule)), "accepted_llm_only": int(len(only_llm)),
           "both_rule_removal_changes_outcome": float(both.dec_rule.mean()) if len(both) else None,
           "both_llm_removal_changes_outcome": float(both.dec_llm.mean()) if len(both) else None,
           "accepted_ge2_stories": float((acc.n_stories >= 2).mean()) if len(acc) else None,
           "constraint_actions": dict(logs)}
json.dump(summary, open(f"{OUT}/explanation_stats_TEST.json", "w", encoding="utf-8"), indent=1)

decisions explained: 704
status     adjusted  final  low  rejected
type                                     
attribute         0      0   74         0
class             0     37  251         0
relation         10      0  331         1

consistency check passed (true by construction, sanity only): 1.0
evidence spans per decision: mean 2.52 | median 1.0

accepted elements: 47
  evidence from both paths: 47 | rules only: 0 | LLM only: 0
  among 'both': removing the RULE evidence changes the outcome in 100%
  among 'both': removing the LLM evidence changes the outcome in 100%
  supported by >= 2 stories: 100% | by >= 5 stories: 51%

constraint-engine actions on TEST:
      9  no cardinality evidence -> conservative default 0..*
      1  hint '..' not valid -> default 0..*
      1  endpoint '..' is a low-confidence class (score ..) -> rejected

trace book reverse lookup (sampled; consistent by construction):
  planningpoker  200 records | 200/200 sampled spans resolve to their element
  cam

In [ ]:
# [21] Reproducibility export: environment + file listing (publish the whole folder with the paper)
import sys, importlib.metadata as md

env = {"python": sys.version.split()[0]}
for p in ("spacy", "transformers", "torch", "networkx", "pandas", "accelerate"):
    try:
        env[p] = md.version(p)
    except Exception:
        env[p] = None
try:
    env["spacy_model"] = f"{nlp.meta['name']} {nlp.meta['version']}"
except Exception:
    env["spacy_model"] = "en_core_web_sm"
env["llm_model"] = MODEL
if os.path.exists("REJ-main.zip"):
    env["benchmark_zip_md5"] = hashlib.md5(open("REJ-main.zip", "rb").read()).hexdigest()
json.dump(env, open(f"{OUT}/environment.json", "w", encoding="utf-8"), indent=1)
print(json.dumps(env, indent=1))
for f in sorted(os.listdir(OUT)):
    print(f"{os.path.getsize(os.path.join(OUT, f)):>10}  {f}")

{
 "python": "3.13.15",
 "spacy": "3.8.16",
 "transformers": "5.18.0",
 "torch": "2.11.0+cu130",
 "networkx": "3.7",
 "pandas": "2.2.3",
 "accelerate": "1.15.0",
 "spacy_model": "core_web_sm 3.8.0",
 "llm_model": "Qwen/Qwen2.5-7B-Instruct",
 "benchmark_zip_md5": "f628b349b6f9cb9f6f196fce5d35a899"
}
     21014  TEST_RESULTS.json
       299  environment.json
       602  explanation_stats_TEST.json
      1478  frozen_config.json
    914059  llm_cache.jsonl
    108919  tracebook_camperplus.jsonl
     83896  tracebook_grocery.jsonl
    111589  tracebook_planningpoker.jsonl
     90352  tracebook_supermarket.jsonl


In [ ]:
# [22] SUPPLEMENTARY (post hoc): score every grid configuration on all 9 projects (cached on Drive)
import numpy as np

ALL = sorted(BM)
for pid in ALL:
    if pid not in POOL:
        build_pool(pid)          # LLM answers come from the Drive cache
_MERGED.clear()

VARIANTS = {"rules + scoring": dict(use_rule=True,  use_llm=False),
            "LLM + scoring":   dict(use_rule=False, use_llm=True),
            "EXDOM (both)":    dict(use_rule=True,  use_llm=True)}

def full_grid(flags, pids):
    snap = copy.deepcopy(CONFIG)
    cfg = dict(include_gray=False, use_constraints=True, **flags)
    rows = []
    for vals in itertools.product(*GRID.values()):
        c = dict(zip(GRID, vals))
        apply_cfg(c)
        for pid in pids:
            rows.append({**c, "pid": pid, **eval_macro([pid], cfg)})
    CONFIG.clear(); CONFIG.update(snap)
    return pd.DataFrame(rows)

GRIDS = {}
t0 = time.time()
for name, flags in VARIANTS.items():
    path = f"{OUT}/grid9_{key_of(name)}.csv"
    if os.path.exists(path):
        GRIDS[name] = pd.read_csv(path)
    else:
        GRIDS[name] = full_grid(flags, ALL)
        GRIDS[name].to_csv(path, index=False)
    print(f"{name:16} grid rows: {len(GRIDS[name])} | {(time.time()-t0)/60:.1f} min", flush=True)

rules + scoring  grid rows: 2268 | 0.6 min
LLM + scoring    grid rows: 2268 | 0.7 min
EXDOM (both)     grid rows: 2268 | 1.2 min


In [ ]:
# [23] SUPPLEMENTARY (post hoc): leave-one-project-out over all 9 projects.
# For each held-out project, each scored variant is tuned on the other 8 (argmax of macro class F1, relaxed).
# Decision rule fixed before running: a difference is called supported only if the two-sided exact sign test
# gives p < 0.05 after Holm correction over the two primary comparisons (EXDOM vs rules + scoring, EXDOM vs LLM + scoring).
from math import comb
from scipy.stats import wilcoxon

cfg_cols = list(GRID)
METRICS = ("cls_P", "cls_R", "cls_F1", "asc_P", "asc_R", "asc_F1")
ORDER = ["A1 rules (raw)", "A3 LLM (raw)", "rules + scoring", "LLM + scoring", "EXDOM (both)"]
SEL, rows = {}, []
for name in VARIANTS:
    df = GRIDS[name]
    piv = df.pivot_table(index=cfg_cols, columns="pid", values="cls_F1")
    idx = df.set_index(cfg_cols + ["pid"]).sort_index()
    for held in ALL:
        train = [p for p in ALL if p != held]
        best = piv[train].mean(axis=1).idxmax()
        r = idx.loc[tuple(best) + (held,)]
        SEL[(name, held)] = dict(zip(cfg_cols, (float(b) for b in best)))
        rows.append({"condition": name, "pid": held, **{k: float(r[k]) for k in METRICS}})
for name in ("A1 rules (raw)", "A3 LLM (raw)"):
    for pid in ALL:
        r = eval_macro([pid], CONDITIONS[name])
        rows.append({"condition": name, "pid": pid, **{k: float(r[k]) for k in METRICS}})
res = pd.DataFrame(rows)

print("== macro over 9 projects, relaxed (each variant tuned leave-one-project-out) ==")
print(res.groupby("condition", sort=False)[list(METRICS)].mean().reindex(ORDER).round(3).to_string())

pv = res.pivot(index="condition", columns="pid", values="cls_F1").reindex(ORDER)
print("\n== per-project class F1 (relaxed) ==")
print(pv.round(3).to_string())
print("\nmean over the 5 DEV projects :", pv[DEV].mean(axis=1).round(3).to_dict())
print("mean over the 4 TEST projects:", pv[TEST].mean(axis=1).round(3).to_dict(),
      "(one-shot TEST run above used a configuration tuned on DEV only)")

def sign_p(w, l):
    n = w + l
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(comb(n, i) for i in range(min(w, l) + 1)) / 2 ** n)

def boot_ci(d, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    d = np.asarray(d, dtype=float)
    means = [rng.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(means, [2.5, 97.5])

def holm(ps):
    order = sorted(range(len(ps)), key=lambda i: ps[i])
    m, adj, run = len(ps), [None] * len(ps), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (m - rank) * ps[i]))
        adj[i] = run
    return adj

crow = []
for other in ORDER[:-1]:
    d = (pv.loc["EXDOM (both)"] - pv.loc[other]).values
    w, l, t = int((d > 1e-9).sum()), int((d < -1e-9).sum()), int((np.abs(d) <= 1e-9).sum())
    try:
        pw = float(wilcoxon(d).pvalue)
    except ValueError:
        pw = float("nan")
    lo, hi = boot_ci(d)
    crow.append({"EXDOM vs": other, "mean_diff": d.mean(), "boot95_lo": lo, "boot95_hi": hi,
                 "wins": w, "ties": t, "losses": l, "p_sign": sign_p(w, l), "p_wilcoxon": pw})
cmp = pd.DataFrame(crow)
prim = cmp["EXDOM vs"].isin(["rules + scoring", "LLM + scoring"])
cmp.loc[prim, "p_sign_holm"] = holm(cmp.loc[prim, "p_sign"].tolist())
print("\n== paired comparison of class F1 across 9 projects (primary = rules + scoring, LLM + scoring) ==")
print(cmp.round(3).to_string(index=False))

# strict matching, same selected configurations
snap = copy.deepcopy(CONFIG)
srows = []
for name, flags in VARIANTS.items():
    for held in ALL:
        apply_cfg(SEL[(name, held)])
        r = eval_macro([held], dict(include_gray=False, use_constraints=True, **flags), relaxed=False)
        srows.append({"condition": name, "pid": held, "cls_F1": r["cls_F1"]})
CONFIG.clear(); CONFIG.update(snap)
for name in ("A1 rules (raw)", "A3 LLM (raw)"):
    for pid in ALL:
        srows.append({"condition": name, "pid": pid,
                      "cls_F1": eval_macro([pid], CONDITIONS[name], relaxed=False)["cls_F1"]})
sres = pd.DataFrame(srows)
print("\n== macro class F1, strict matching ==")
print(sres.groupby("condition", sort=False)["cls_F1"].mean().reindex(ORDER).round(3).to_string())

print("\n== EXDOM configuration selected in each fold ==")
print(pd.DataFrame({h: SEL[("EXDOM (both)", h)] for h in ALL}).T.to_string())

# optional: results by benchmark source. Fill in from the benchmark paper which three collections come from the
# public repository, e.g. REPO_PROJECTS = ["grocery", "supermarket", "school"]  (this is only an example format)
REPO_PROJECTS = None
if REPO_PROJECTS:
    other = [p for p in ALL if p not in REPO_PROJECTS]
    print("\n== class F1 by source ==")
    print(pd.DataFrame({"repository": pv[REPO_PROJECTS].mean(axis=1),
                        "student projects": pv[other].mean(axis=1)}).round(3).to_string())

json.dump({"selected_configs": {f"{n}|{h}": c for (n, h), c in SEL.items()},
           "per_project_class_F1_relaxed": pv.round(4).to_dict(),
           "paired_comparisons": cmp.round(4).to_dict("records"),
           "strict_macro_class_F1": sres.groupby("condition", sort=False)["cls_F1"].mean().round(4).to_dict()},
          open(f"{OUT}/LOPO9_RESULTS.json", "w", encoding="utf-8"), indent=1)
print("\nsaved ->", f"{OUT}/LOPO9_RESULTS.json")

== macro over 9 projects, relaxed (each variant tuned leave-one-project-out) ==
                 cls_P  cls_R  cls_F1  asc_P  asc_R  asc_F1
condition                                                  
A1 rules (raw)   0.151  0.916   0.254  0.063  0.275   0.099
A3 LLM (raw)     0.302  0.665   0.407  0.110  0.265   0.151
rules + scoring  0.407  0.526   0.444  0.000  0.000   0.000
LLM + scoring    0.501  0.470   0.472  0.111  0.014   0.025
EXDOM (both)     0.547  0.554   0.532  0.050  0.027   0.033

== per-project class F1 (relaxed) ==
pid              camperplus  fishchips  grocery  planningpoker  recycling  school  sports  supermarket  ticket
condition                                                                                                     
A1 rules (raw)        0.375      0.123    0.261          0.160      0.148   0.333   0.400        0.250   0.237
A3 LLM (raw)          0.490      0.160    0.491          0.500      0.244   0.471   0.497        0.457   0.354
rules + scoring   

In [ ]:
# [24] Human rating sheets for the explanations (TEST, final EXDOM configuration)
import openpyxl
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment

EXPL_RUBRIC = """EXPLANATION RATING - INSTRUCTIONS (about 45-60 minutes)
You rate automatically generated explanations of modeling decisions (classes, attributes, relations) extracted from user stories.
Work alone. Do not discuss your ratings with the other rater and do not look at any other file.

For every row of the sheet 'Ratings', read the explanation (column B) and the cited user stories (column C), then give three scores:

clarity (0-2): 0 = vague or confusing, key terms undefined. 1 = the claim is clear but part of the text is unclear or too technical. 2 = precise and understandable without knowing the system.
evidence_supports (0-2): do the quoted spans really support the claim made in the first sentence? 0 = no or unrelated. 1 = partly (related words, but they do not establish the element). 2 = yes, clearly.
actionability (0-2): rate it ONLY if the explanation suggests a next step (typically elements that were set aside or rejected). Leave it EMPTY if no next step is given (for example accepted elements). 0 = the step is wrong or misleading. 1 = generic ('find more evidence'). 2 = specific and feasible.

Legend. R0 = the role named after 'As a ...'. R1 = a noun used as subject or object. R2 = a noun-noun compound (the modifier is a class, the head noun is its attribute). R3 = a transitive verb linking a subject and an object. 'LLM suggestion with verified quote' = a language model proposed the element and quoted text that really occurs in the story. Score zones: accept >= 0.60, gray 0.30-0.60, low < 0.30. 'Counterfactual' = the score the system computes when one kind of evidence is removed.
Judge only what the explanation and the stories show. Do not use outside knowledge about the 'right' model.
Use the comment column for anything unclear. Save the file under a new name ending in _filled.xlsx."""

QUOTAS = {("class", "final"): 14, ("class", "adjusted"): 1, ("class", "low"): 14,
          ("relation", "adjusted"): 9, ("relation", "low"): 14, ("attribute", "low"): 8}

def write_workbook(path, rubric, sheets, validations=None, widths=None):
    wrap = Alignment(wrap_text=True, vertical="top")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        pd.DataFrame({"instructions": rubric.strip().splitlines()}).to_excel(
            xw, sheet_name="Instructions", index=False)
        xw.sheets["Instructions"].column_dimensions["A"].width = 150
        for name, d in sheets.items():
            d.to_excel(xw, sheet_name=name, index=False)
            ws = xw.sheets[name]
            for col, w in (widths or {}).get(name, {}).items():
                ws.column_dimensions[col].width = w
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.alignment = wrap
            for col in (validations or {}).get(name, []):
                dv = DataValidation(type="list", formula1='"0,1,2"', allow_blank=True)
                ws.add_data_validation(dv)
                dv.add(f"{col}2:{col}{len(d) + 1}")

def build_explanation_items(pids, seed=7):
    snap = copy.deepcopy(CONFIG)
    apply_cfg(FROZEN["EXDOM (both)"]["config"])
    strata = collections.defaultdict(list)
    for pid in pids:
        texts = PROJECTS[pid]["stories"]
        rule, llm, _ = POOL[pid]
        merged_ = merge(rule + llm)
        for c in merged_:
            score(c)
        set_margins(merged_)
        for c in decide(merged_, texts, include_gray=False, use_constraints=True):
            strata[(c.type, c.status)].append((pid, c, texts))
    rng = random.Random(seed)
    chosen = []
    for k_, quota in QUOTAS.items():
        items = strata.get(k_, [])
        chosen += rng.sample(items, min(quota, len(items)))
    rng.shuffle(chosen)
    rows, key_rows = [], []
    for i, (pid, c, texts) in enumerate(chosen, 1):
        e = explain_b1(c, texts)
        cited = pick_spans(c)
        sids = []
        for s in cited:
            if s.story_id not in sids:
                sids.append(s.story_id)
        rid = f"E{i:03d}"
        rows.append({"id": rid, "explanation": e["text"],
                     "stories": "\n".join(f"[{s}] {texts[s]}" for s in sids)})
        key_rows.append({"id": rid, "pid": pid, "type": c.type, "status": c.status,
                         "score": c.score, "n_quotes": len(cited), "n_spans": len(c.spans)})
    CONFIG.clear(); CONFIG.update(snap)
    return pd.DataFrame(rows), pd.DataFrame(key_rows)

expl_df, expl_key = build_explanation_items(TEST)
for tag, seed in (("A", 1), ("B", 2)):
    d = expl_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    for c_ in ("clarity", "evidence_supports", "actionability", "comment"):
        d[c_] = None
    write_workbook(f"{OUT}/explanations_rater_{tag}.xlsx", EXPL_RUBRIC, {"Ratings": d},
                   validations={"Ratings": ["D", "E", "F"]},
                   widths={"Ratings": {"A": 8, "B": 90, "C": 70, "G": 30}})
expl_key.to_csv(f"{OUT}/explanations_key.csv", index=False)       # do NOT share with the raters
open(f"{OUT}/explanations_rater_instructions.md", "w", encoding="utf-8").write(EXPL_RUBRIC)
print("items per stratum:")
print(expl_key.groupby(["type", "status"]).size().to_string())
print("\nGive rater A and rater B ONLY their own file:",
      f"{OUT}/explanations_rater_A.xlsx and explanations_rater_B.xlsx")

items per stratum:
type       status  
attribute  low          8
class      adjusted     1
           final       14
           low         14
relation   adjusted     9
           low         14

Give rater A and rater B ONLY their own file: /content/drive/MyDrive/exdom/explanations_rater_A.xlsx and explanations_rater_B.xlsx


In [ ]:
OUT = "/content/drive/MyDrive/exdom"

In [ ]:
import os, json, re, random, collections, copy
import numpy as np, pandas as pd
OUT = "/content/drive/MyDrive/exdom"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# [25] Analyse the two filled explanation sheets (run after both raters have finished)
def qw_kappa(a, b, k=3):
    a, b = np.asarray(a, dtype=int), np.asarray(b, dtype=int)
    O = np.zeros((k, k))
    for x, y in zip(a, b):
        O[x, y] += 1
    O /= O.sum()
    E = np.outer(O.sum(1), O.sum(0))
    W = np.array([[(i - j) ** 2 for j in range(k)] for i in range(k)]) / (k - 1) ** 2
    den = (W * E).sum()
    return float(1 - (W * O).sum() / den) if den > 0 else float("nan")

def boot_kappa(a, b, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    vals = []
    for _ in range(n_boot):
        ix = rng.integers(0, len(a), len(a))
        v = qw_kappa(a[ix], b[ix])
        if not np.isnan(v):
            vals.append(v)
    return tuple(np.percentile(vals, [2.5, 97.5])) if vals else (float("nan"), float("nan"))

fa, fb = f"{OUT}/explanations_rater_A_filled.xlsx", f"{OUT}/explanations_rater_B_filled.xlsx"
if not (os.path.exists(fa) and os.path.exists(fb)):
    print("Waiting for the filled sheets. Save them as:\n ", fa, "\n ", fb)
else:
    ra = pd.read_excel(fa, sheet_name="Ratings").set_index("id")
    rb = pd.read_excel(fb, sheet_name="Ratings").set_index("id")
    key_ = pd.read_csv(f"{OUT}/explanations_key.csv").set_index("id")
    out, dims = {}, ["clarity", "evidence_supports", "actionability"]
    for dim in dims:
        both = pd.concat([ra[dim], rb[dim]], axis=1, keys=["A", "B"]).dropna()
        assert set(np.unique(both.values)) <= {0, 1, 2}, f"{dim}: values must be 0, 1 or 2"
        both = both.astype(int)
        lo, hi = boot_kappa(both.A, both.B)
        out[dim] = {"n": int(len(both)), "mean_A": float(both.A.mean()), "mean_B": float(both.B.mean()),
                    "exact_agreement": float((both.A == both.B).mean()),
                    "weighted_kappa": qw_kappa(both.A, both.B), "kappa_ci95": [lo, hi]}
        print(f"{dim:18} n={len(both):3} meanA={both.A.mean():.2f} meanB={both.B.mean():.2f} "
              f"exact agreement={(both.A == both.B).mean():.2f} "
              f"quadratic-weighted kappa={out[dim]['weighted_kappa']:.2f} [{lo:.2f}, {hi:.2f}]")
    mean_rating = pd.concat([ra[dims], rb[dims]]).groupby(level=0).mean()
    res_e = key_.join(mean_rating)
    print("\nmean of the two raters by stratum:")
    print(res_e.groupby(["type", "status"])[dims].mean().round(2).to_string())
    print("\nautomatic criterion (paper rubric): explanations quoting >= 2 evidence spans:",
          f"{(key_.n_quotes >= 2).mean():.0%}",
          "| among accepted elements:",
          f"{(key_[key_.status.isin(['final', 'adjusted'])].n_quotes >= 2).mean():.0%}")
    json.dump({"per_dimension": out}, open(f"{OUT}/explanation_ratings_summary.json", "w"), indent=1)

clarity            n= 60 meanA=0.97 meanB=1.93 exact agreement=0.03 quadratic-weighted kappa=0.06 [0.00, 0.13]
evidence_supports  n= 60 meanA=1.43 meanB=1.65 exact agreement=0.65 quadratic-weighted kappa=0.40 [0.20, 0.59]
actionability      n= 36 meanA=2.00 meanB=2.00 exact agreement=1.00 quadratic-weighted kappa=nan [nan, nan]

mean of the two raters by stratum:
                    clarity  evidence_supports  actionability
type      status                                             
attribute low          1.38               1.56            2.0
class     adjusted     1.50               2.00            NaN
          final        1.50               1.93            NaN
          low          1.50               1.21            2.0
relation  adjusted     1.50               1.56            NaN
          low          1.36               1.43            2.0

automatic criterion (paper rubric): explanations quoting >= 2 evidence spans: 58% | among accepted elements: 100%


In [ ]:
# [26] Sheets for manual validation of the automatic name matching (EXDOM predictions, leave-one-out configs)
MATCH_RUBRIC = """NAME MATCHING JUDGMENTS - INSTRUCTIONS (about 1-2 hours)
An automatic system proposed class names for software projects (extracted from user stories). A reference model (gold standard) exists for each project.
You judge whether names correspond. Work alone and do not look at any other file.

Two names CORRESPOND if a domain expert would treat them as the same concept in this project: singular or plural, synonyms, or the same concept written with more or fewer words (for example 'payment method' and 'payment methods'). They do NOT correspond if the concepts are related but different (for example 'order' and 'payment'), or if one is clearly more general or more specific than the other.

Sheet 'Predicted': for each predicted class, look at column 'gold_classes' and write EXACTLY the gold name it corresponds to, or NONE.
Sheet 'Gold': for each gold class, look at column 'predicted_classes' and write EXACTLY the predicted name that covers it, or NONE.
Type the name exactly as it appears in the list (capitalization does not matter). Use column 'story_context' only to understand the meaning of a predicted name.
Save the file under a new name ending in _filled.xlsx."""

sel_all = json.load(open(f"{OUT}/LOPO9_RESULTS.json"))["selected_configs"]

def exdom_predictions(pid):
    snap = copy.deepcopy(CONFIG)
    apply_cfg(sel_all[f"EXDOM (both)|{pid}"])
    model_, _ = run_condition(pid, use_rule=True, use_llm=True, include_gray=False, use_constraints=True)
    CONFIG.clear(); CONFIG.update(snap)
    out = {}
    for c in model_:
        if c.type == "class":
            out[c.name] = PROJECTS[pid]["stories"][c.spans[0].story_id]
    return out

prow, grow, pkey, gkey = [], [], [], []
for pid in ALL:
    gold = PROJECTS[pid]["gold"]["classes"]
    preds = exdom_predictions(pid)
    pnames = list(preds)
    for n, snip in preds.items():
        i = f"P{len(prow) + 1:03d}"
        prow.append({"id": i, "project": pid, "predicted_class": n, "story_context": snip,
                     "gold_classes": " | ".join(gold), "matches_gold": None, "comment": None})
        pkey.append({"id": i, "pid": pid,
                     "auto_strict": any(match_names(n, g, False) for g in gold),
                     "auto_relaxed": any(match_names(n, g, True) for g in gold)})
    for g in gold:
        i = f"G{len(grow) + 1:03d}"
        grow.append({"id": i, "project": pid, "gold_class": g,
                     "predicted_classes": " | ".join(pnames) or "(none)", "covered_by": None, "comment": None})
        gkey.append({"id": i, "pid": pid,
                     "auto_strict": any(match_names(n, g, False) for n in pnames),
                     "auto_relaxed": any(match_names(n, g, True) for n in pnames)})
P_df, G_df = pd.DataFrame(prow), pd.DataFrame(grow)
pd.concat([pd.DataFrame(pkey), pd.DataFrame(gkey)]).to_csv(f"{OUT}/matching_key.csv", index=False)   # keep private

rng = random.Random(11)
subP = set(rng.sample(list(P_df.id), max(1, round(0.3 * len(P_df)))))
subG = set(rng.sample(list(G_df.id), max(1, round(0.3 * len(G_df)))))
wid = {"Predicted": {"A": 7, "B": 14, "C": 28, "D": 70, "E": 70, "F": 28, "G": 25},
       "Gold": {"A": 7, "B": 14, "C": 28, "D": 70, "E": 28, "F": 25}}
write_workbook(f"{OUT}/matching_rater_A.xlsx", MATCH_RUBRIC,
               {"Predicted": P_df.sample(frac=1, random_state=3).reset_index(drop=True),
                "Gold": G_df.sample(frac=1, random_state=3).reset_index(drop=True)}, widths=wid)
write_workbook(f"{OUT}/matching_rater_B.xlsx", MATCH_RUBRIC,
               {"Predicted": P_df[P_df.id.isin(subP)].sample(frac=1, random_state=4).reset_index(drop=True),
                "Gold": G_df[G_df.id.isin(subG)].sample(frac=1, random_state=4).reset_index(drop=True)}, widths=wid)
open(f"{OUT}/matching_rater_instructions.md", "w", encoding="utf-8").write(MATCH_RUBRIC)
print(f"rater A: {len(P_df)} predicted + {len(G_df)} gold rows | rater B (30% subset): {len(subP)} + {len(subG)} rows")

rater A: 92 predicted + 96 gold rows | rater B (30% subset): 28 + 29 rows


In [ ]:
# [27] Analyse the filled matching sheets: human-judged precision/recall versus the automatic matching
def kappa_bin(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float((po - pe) / (1 - pe)) if pe < 1 else float("nan")

def judge(value, options):
    v = str(value).strip().lower() if pd.notna(value) else ""
    if v in ("", "nan"):
        return None
    if v == "none":
        return False
    return True if v in {o.strip().lower() for o in options.split("|")} else "invalid"

def read_judged(path):
    p = pd.read_excel(path, sheet_name="Predicted")
    g = pd.read_excel(path, sheet_name="Gold")
    p["human"] = [judge(v, o) for v, o in zip(p["matches_gold"], p["gold_classes"])]
    g["human"] = [judge(v, o) for v, o in zip(g["covered_by"], g["predicted_classes"])]
    return p.set_index("id"), g.set_index("id")

fA, fB = f"{OUT}/matching_rater_A_filled.xlsx", f"{OUT}/matching_rater_B_filled.xlsx"
if not os.path.exists(fA):
    print("Waiting for rater A's filled sheet:", fA)
else:
    mk = pd.read_csv(f"{OUT}/matching_key.csv").set_index("id")
    pA, gA = read_judged(fA)
    for nm, d in (("Predicted", pA), ("Gold", gA)):
        bad = d[d.human == "invalid"]
        miss = d[d.human.isna()]
        print(f"{nm}: {len(d)} rows | unanswered {len(miss)} | not an exact name from the list {len(bad)}")
        if len(bad):
            print(bad.iloc[:, :3].to_string())
    ok_p = pA[pA.human.isin([True, False])].join(mk[["auto_strict", "auto_relaxed"]])
    ok_g = gA[gA.human.isin([True, False])].join(mk[["auto_strict", "auto_relaxed"]])

    def macro_prf(dp, dg, col):
        f1s, Ps, Rs = [], [], []
        for pid in ALL:
            p_ = dp[dp.project == pid][col].astype(bool)
            g_ = dg[dg.project == pid][col].astype(bool)
            P_ = p_.mean() if len(p_) else 0.0
            R_ = g_.mean() if len(g_) else 0.0
            Ps.append(P_); Rs.append(R_)
            f1s.append(2 * P_ * R_ / (P_ + R_) if P_ + R_ else 0.0)
        return round(float(np.mean(Ps)), 3), round(float(np.mean(Rs)), 3), round(float(np.mean(f1s)), 3)

    rows_ = {"automatic strict": macro_prf(ok_p, ok_g, "auto_strict"),
             "automatic relaxed": macro_prf(ok_p, ok_g, "auto_relaxed"),
             "human (rater A)": macro_prf(ok_p, ok_g, "human")}
    print("\nEXDOM class extraction, macro over projects (P, R, F1):")
    print(pd.DataFrame(rows_, index=["P", "R", "F1"]).T.to_string())

    summ = {"macro_PRF1": {k: list(v) for k, v in rows_.items()}}
    for nm, d in (("predicted side", ok_p), ("gold side", ok_g)):
        h = d.human.astype(bool)
        for col in ("auto_strict", "auto_relaxed"):
            a = d[col].astype(bool)
            cm = {"auto_yes_human_no": int((a & ~h).sum()), "auto_no_human_yes": int((~a & h).sum())}
            print(f"{nm:15} {col:13} agreement={(a == h).mean():.2f} kappa={kappa_bin(a, h):.2f} {cm}")
            summ[f"{nm}|{col}"] = {"agreement": float((a == h).mean()), "kappa": kappa_bin(a, h), **cm}
    if os.path.exists(fB):
        pB, gB = read_judged(fB)
        a_ = pd.concat([pA.human.reindex(pB.index), gA.human.reindex(gB.index)])
        b_ = pd.concat([pB.human, gB.human])
        m = a_.isin([True, False]) & b_.isin([True, False])
        print(f"\ninter-rater agreement A vs B on {int(m.sum())} items: "
              f"agreement={(a_[m].astype(bool) == b_[m].astype(bool)).mean():.2f} "
              f"kappa={kappa_bin(a_[m], b_[m]):.2f}")
        summ["inter_rater"] = {"n": int(m.sum()), "kappa": kappa_bin(a_[m], b_[m])}
    json.dump(summ, open(f"{OUT}/matching_validation_summary.json", "w"), indent=1)

Predicted: 92 rows | unanswered 0 | not an exact name from the list 0
Gold: 96 rows | unanswered 0 | not an exact name from the list 0

EXDOM class extraction, macro over projects (P, R, F1):
                       P      R     F1
automatic strict   0.480  0.481  0.463
automatic relaxed  0.511  0.508  0.492
human (rater A)    0.508  0.487  0.478
predicted side  auto_strict   agreement=0.97 kappa=0.93 {'auto_yes_human_no': 0, 'auto_no_human_yes': 3}
predicted side  auto_relaxed  agreement=0.98 kappa=0.96 {'auto_yes_human_no': 1, 'auto_no_human_yes': 1}
gold side       auto_strict   agreement=0.99 kappa=0.98 {'auto_yes_human_no': 0, 'auto_no_human_yes': 1}
gold side       auto_relaxed  agreement=0.96 kappa=0.92 {'auto_yes_human_no': 3, 'auto_no_human_yes': 1}

inter-rater agreement A vs B on 57 items: agreement=0.82 kappa=0.65


In [ ]:
   print(ra.drop(columns=[c for c in ra.columns if ra[c].dtype==object], errors="ignore").apply(lambda s: s.value_counts()).fillna(0).astype(int))
   print(rb.drop(columns=[c for c in rb.columns if rb[c].dtype==object], errors="ignore").apply(lambda s: s.value_counts()).fillna(0).astype(int))

     clarity  evidence_supports  actionability  comment
0.0        2                  2              0        0
1.0       58                 30              0        0
2.0        0                 28             36        0
   clarity  evidence_supports  actionability
2       60                 60             36


In [ ]:
d = ra[["evidence_supports","clarity"]].join(key_[["type","status"]], how="left")
print(d.groupby(["type","status"])["evidence_supports"].value_counts().unstack(fill_value=0))

evidence_supports   0   1   2
type      status             
attribute low       0   3   5
class     adjusted  0   0   1
          final     0   1  13
          low       0  12   2
relation  adjusted  0   7   2
          low       2   7   5


In [ ]:
from transformers import GenerationConfig
for m in ("Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"):
    g = GenerationConfig.from_pretrained(m)
    print(m, {k: getattr(g, k, None) for k in
          ("temperature", "top_p", "top_k", "repetition_penalty", "do_sample")})

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen/Qwen2.5-3B-Instruct {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20, 'repetition_penalty': 1.05, 'do_sample': True}


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen/Qwen2.5-7B-Instruct {'temperature': 0.7, 'top_p': 0.8, 'top_k': 20, 'repetition_penalty': 1.05, 'do_sample': True}


In [ ]:
import sys, importlib.metadata as md
for p in ("spacy","transformers","torch","networkx","pandas","accelerate","bitsandbytes"):
    try: print(p, md.version(p))
    except Exception: print(p, None)
print("python", sys.version.split()[0])

spacy 3.8.16
transformers 5.18.0
torch 2.11.0+cu130
networkx 3.7
pandas 2.2.3
accelerate 1.15.0
bitsandbytes None
python 3.13.15


In [ ]:
!pip -q install bitsandbytes
import importlib.metadata as m; print(m.version("bitsandbytes"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.3 MB/s eta 0:00:00
0.50.2


In [ ]:
import importlib.metadata as m; print(m.version("bitsandbytes"))

0.50.2


In [ ]:
import glob, os, re
base = "REJ/REJ-main/A Benchmark for Domain Model/User Stories"
DEV = ['fishchips', 'sports', 'school', 'ticket', 'recycling']
n = 0
for f in glob.glob(f"{base}/*.txt"):
    k = re.sub(r"[^a-z0-9]", "", os.path.splitext(os.path.basename(f))[0].lower())
    if k in DEV:
        c = sum(1 for l in open(f, encoding="utf-8-sig", errors="replace") if l.strip())
        print(k, c); n += c
print("DEV stories:", n, "-> LLM runs:", 2*n)

fishchips 50
school 61
recycling 51
ticket 54
sports 63
DEV stories: 279 -> LLM runs: 558


In [ ]:
for pid in TEST:
    build_pool(pid)      # از کش می‌خواند؛ هیچ تولید جدیدی انجام نمی‌شود
llm_report(TEST)

{'proposed': 1191, 'malformed': 0, 'uncertain': 71, 'invalid_evidence': 24, 'ungrounded': 70, 'parse_fail': 70, 'kept': 462}
invalid-evidence rate: 0.02 | ungrounded-name rate: 0.059
